In [462]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import HistGradientBoostingRegressor
import librosa
from sklearn.pipeline import make_pipeline
from scipy.stats import skew, kurtosis
from scipy.signal import hilbert
from math import log10
import parselmouth
from maad import sound as suono
from maad import features as ft
import pywt

In [463]:
csvTotale = pd.read_csv('.././data/development.csv', header=0, index_col=0).drop(columns=['sampling_rate'])
csvTotale['gender'] = csvTotale['gender'].map(lambda x: 0 if x=='male' else 1)
csvTotale['tempo'] = csvTotale['tempo'].map(lambda x: float(x[1:-1]))
csvTotale['path'] = csvTotale['path'].map(lambda x: x.split('/')[-1])

In [464]:
csvEval = pd.read_csv('.././data/evaluation.csv', header=0, index_col=0).drop(columns=['sampling_rate'])
csvEval['gender'] = csvEval['gender'].map(lambda x: 0 if x=='male' else 1)
csvEval['tempo'] = csvEval['tempo'].map(lambda x: float(x[1:-1]))
csvEval['path'] = csvEval['path'].map(lambda x: x.split('/')[-1])

In [465]:
cross_val_score(HistGradientBoostingRegressor(categorical_features=csvTotale.drop(columns=['path', 'age']).dtypes == 'object'), 
                csvTotale.drop(columns=['path', 'age']), 
                csvTotale['age'], cv=10, scoring='neg_mean_absolute_error').mean()

# BASELINE = -7.24

np.float64(-7.242994400990241)

In [466]:
audioDevelopment = {file: librosa.load('.././data/audios_development/'+file, sr=22050)[0] for file in csvTotale['path']}

In [467]:
def getMFCC(audio):  # Keep imp=1 otherwise the score worsens
    numFcc = 35
    return pd.Series(librosa.feature.mfcc(y=audio, sr=22050, n_mfcc=numFcc).mean(axis=1), index=[f'mfcc_mean{i}' for i in range(numFcc)])

In [468]:
def getSpectralEnergy(audio):
    S = librosa.stft(audio)
    freqs = librosa.fft_frequencies(sr=22050)
    return pd.Series([np.sum(np.abs(S[(freqs >= 150) & (freqs <= 650)])**2), np.sum(np.abs(S[(freqs >= 1000) & (freqs <= 8000)])**2)],
                        index=['spectralEnergy250-650', 'spectralEnergy1000-8000'])

In [469]:
def computeF0(audio):
    sound = parselmouth.Sound(audio)
    pitch = sound.to_pitch()
    info = str(pitch.info()).split('\n')
    return pd.Series([pitch.count_voiced_frames(), pitch.get_mean_absolute_slope(), pitch.xmax-pitch.xmin, 
                      pitch.n_frames, *[float(info[15+i].split('=')[2].lstrip().split(' =')[0].split()[0]) for i in range(0, 5)],
                      *[float(info[21+i].split('=')[2].lstrip().split(' =')[0].split()[0]) for i in range(0, 3)]],
                     index=['nVoicedFrames', 'meanAbsoluteSlope', 'duration', 'nFrames', 
                            'q10', 'q16', 'q50', 'q84', 'q90', '84-median', 'median-16', '90-10']
                     )

In [470]:
def computeMath(audio):
    return pd.Series([skew(audio), kurtosis(audio), np.mean(np.abs(hilbert(audio))), np.mean(np.abs(np.fft.fft(audio)))],
                     index=['skew', 'kurtosis', 'hilbertMean', 'fftMean'])

In [471]:
def computSNR(audio):
    return pd.Series([suono.temporal_snr(audio)[-1]], index=['temporalSNR'])

In [472]:
def computeTemporalMedia(audio):
    return pd.Series([ft.temporal_median(audio)], index=['temporalMedian'])

In [473]:
def computeAllFeatures(audio):
    return pd.Series(ft.all_temporal_features(audio, fs=22050, nperseg=256).values[0],
                     index=['sm', 'sv', 'ss', 'sk', 'Time 5%', "Time 25%", "Time 50%", "Time 75%", "Time 95%  ", 
                            "zcr", "duration_5", "duration_90"])

In [474]:
def computePeakFrequency(audio):
    return pd.Series(ft.peak_frequency(audio, fs=22050, nperseg=256, amp=True),
                     index=['peakFrequency', 'peakFrequencyAmp'])

In [475]:
def computeEntropy(audio):
    return pd.Series(ft.temporal_entropy(audio), index=['entropy'])

In [476]:
def getLogMel(audio):
    return pd.Series(librosa.amplitude_to_db(librosa.feature.melspectrogram(y=audio, sr=22050, n_fft=256, hop_length=512, n_mels=16).mean(axis=1)),
                     index=[f'mel{i}' for i in range(16)])

In [477]:
def computeRMS(audio):
    rms = librosa.feature.rms(y=audio)
    return pd.Series([20*log10(rms.mean()), rms.std(), rms.max(), 20*log10(rms.sum())], index=['rms_mean', 'rms_std', 'rms_max', 'rms_sum'])

In [478]:
def getOnsetRate(audio):
    return pd.Series(librosa.onset.onset_detect(y=audio, sr=22050, hop_length=512).shape[0]/audio.shape[0], index=['onsetRate'])

In [479]:
def getPitchStats(audio):
    _, flag, prob = librosa.pyin(y=audio, fmin=50, fmax=2000, sr=22050)
    return pd.Series([flag[flag].shape[0], np.mean(prob), np.std(prob)], index=['FrameVoice', 'meanProb', 'stdProb'])

In [480]:
# FUCKING GOATED
def getPausesAndArticulationRate(audio):
    intervals = librosa.effects.split(audio, top_db=20)
    total_duration = len(audio) / 22050
    speech_duration = sum((end - start) for start, end in intervals) / 22050
    return pd.Series([total_duration - speech_duration, speech_duration / total_duration], index=['pauses', 'articulation_rate'])

## WAVELET

### ENERGY

In [481]:
def compute_energy_distribution(coefficients):
    return np.array([np.sum(np.abs(c)**2) for c in coefficients])

### ENTROPY

In [482]:
def compute_wavelet_entropy(coefficients):
    energy = compute_energy_distribution(coefficients)
    return pd.Series(-sum(p * np.log2(p) for p in energy / np.sum(energy) if p > 1e-12), index=['wavelet_entropy'])

### DOMINANT SCALES

In [483]:
def find_dominant_scales(coefficients, scales, num_scales=10):
    energy = compute_energy_distribution(coefficients)
    return pd.Series(np.mean([scales[i] for i in np.argsort(energy)[-num_scales:]]), index=['avg_dominant_scale'])

### AMPLITUDE AND FREQUENCY MODULATION

In [484]:
def compute_amplitude_modulation(coefficients):
    return pd.Series(np.sum(np.abs(coefficients), axis=1).reshape(-1, 4).mean(axis=1), index=[f'amplitude_modulation{i}' for i in range(24)])

### HARMONICS

In [485]:
def analyze_harmonics(coefficients):
    """
    Analyze harmonic content based on coefficient patterns.
    Parameters:
        coefficients (list of ndarray): Wavelet coefficients.
    Returns:
        dict: Harmonic and non-harmonic components.
    """
    harmonic_content = [np.max(np.abs(c)) for c in coefficients.sum(axis=1)]
    return {'harmonic_content': harmonic_content}

###  WAVELET CEPSTRAL

In [486]:
def compute_wavelet_cepstral_features(coefficients):
    """
    Compute cepstral features from wavelet coefficients.
    Parameters:
        coefficients (list of ndarray): Wavelet coefficients.
    Returns:
        ndarray: Cepstral features.
    """
    return pd.Series(np.fft.fft(np.log(np.abs(coefficients)+1e-12), axis=1).real.mean(axis=1).reshape(-1, 4).mean(axis=1),
                     index=[f'cepstral_feature{i}' for i in range(24)])

In [487]:
def wavelet_transform(audio):
    return pywt.cwt(data=audio, wavelet='morl', scales=[
        298.59375,    293.69877049, 288.96169355, 284.375,
        279.93164062, 275.625     , 271.44886364, 267.39738806,
        263.46507353, 259.64673913, 255.9375    , 252.33274648,
        248.828125  , 245.41952055, 242.10304054, 238.875     ,
        235.73190789, 232.67045455, 229.6875    , 226.78006329,
        223.9453125 , 221.18055556, 218.48323171, 215.85090361,
        213.28125   , 210.77205882, 208.32122093, 205.92672414,
        203.58664773, 201.2991573 , 199.0625    , 196.875     ,
        194.73505435, 192.64112903, 190.59175532, 188.58552632,
        186.62109375, 184.69716495, 182.8125    , 180.96590909,
        179.15625   , 177.38242574, 175.64338235, 173.9381068 ,
        172.265625  , 170.625     , 169.01533019, 167.43574766,
        165.88541667, 164.36353211, 162.86931818, 161.40202703,
        159.9609375 , 158.54535398, 157.15460526, 155.78804348,
        154.4450431 , 153.125     , 151.82733051, 150.55147059,
        149.296875  , 148.06301653, 146.84938525, 145.6554878 ,
        144.48084677, 143.325     , 142.1875    , 141.06791339,
        139.96582031, 138.88081395, 137.8125    , 136.76049618,
        135.72443182, 134.70394737, 133.69869403, 132.70833333,
        131.73253676, 130.7709854 , 129.82336957, 128.88938849,
        127.96875   , 127.06117021, 126.16637324, 125.28409091,
        124.4140625 , 123.55603448, 122.70976027, 121.875     ,
        121.05152027, 120.23909396, 119.4375    , 118.64652318,
        117.86595395, 117.09558824, 116.33522727, 115.58467742], sampling_period=1/22050)

def doWavelet(audio):
    coefficients, scales = wavelet_transform(audio)
    
    return pd.concat([compute_wavelet_entropy(coefficients), 
                      find_dominant_scales(coefficients, scales),
                      compute_wavelet_cepstral_features(coefficients)
                      ], axis=0)

In [490]:
def computeMetrics(audios):
    return pd.DataFrame({file: pd.concat([
            getMFCC(audios[file]),
            getSpectralEnergy(audios[file]),
            computeF0(audios[file]),
            computeMath(audios[file]),
            computSNR(audios[file]),
            computeTemporalMedia(audios[file]),
            computeAllFeatures(audios[file]),
            computePeakFrequency(audios[file]),
            computeEntropy(audios[file]),   # fucking entropy drops the score
            # computeSpectralActivity(audios[file]),
            # computeSpectralCover(audios[file]),
            # getSpectralContrast(audios[file]),
            # getOnsetRate(audios[file]),
            # getPitchStats(audios[file]),
            # getPausesAndArticulationRate(audios[file]),
            ], axis=0)
        for file in audios}).T

In [491]:
metrics = pd.concat([computeMetrics(audioDevelopment), 
                    csvTotale.set_index('path')[['hnr', 'shimmer', 'jitter', 'gender', 'max_pitch', 'mean_pitch', 'min_pitch']]], axis=1)

metrics['log_hnr']= metrics['hnr'].map(lambda x: 20*log10(np.abs(x)))

In [435]:
# TIENI GLI LFCC

Index(['mfcc_mean0', 'mfcc_mean1', 'mfcc_mean2', 'mfcc_mean3', 'mfcc_mean4',
       'mfcc_mean5', 'mfcc_mean6', 'mfcc_mean7', 'mfcc_mean8', 'mfcc_mean9',
       'mfcc_mean10', 'mfcc_mean11', 'mfcc_mean12', 'mfcc_mean13',
       'mfcc_mean14', 'mfcc_mean15', 'mfcc_mean16', 'mfcc_mean17',
       'mfcc_mean18', 'mfcc_mean19', 'mfcc_mean20', 'mfcc_mean21',
       'mfcc_mean22', 'mfcc_mean23', 'mfcc_mean24', 'mfcc_mean25',
       'mfcc_mean26', 'mfcc_mean27', 'mfcc_mean28', 'mfcc_mean29',
       'mfcc_mean30', 'mfcc_mean31', 'mfcc_mean32', 'mfcc_mean33',
       'mfcc_mean34', 'spectralEnergy250-650', 'spectralEnergy1000-8000',
       'nVoicedFrames', 'meanAbsoluteSlope', 'duration', 'nFrames', 'q10',
       'q16', 'q50', 'q84', 'q90', '84-median', 'median-16', '90-10', 'skew',
       'kurtosis', 'hilbertMean', 'fftMean', 'temporalSNR', 'temporalMedian',
       'sm', 'sv', 'ss', 'sk', 'Time 5%', 'Time 25%', 'Time 50%', 'Time 75%',
       'Time 95%  ', 'zcr', 'duration_5', 'duration_90', 'pea

In [378]:
def wav2File(audios, file):
    try:
        items = pd.read_csv(file, header=0, index_col=0).index
    except Exception:
        items = set()
    with open(file, 'a+', encoding='utf-8') as fp:
        for key in set(audios.keys()).difference(items):
            fp.write(key+','+','.join(map(str, doWavelet(audios[key]).values))+'\n')

In [492]:
cross_val_score(HistGradientBoostingRegressor(), metrics,
                 csvTotale['age'], cv=20, scoring='neg_mean_absolute_error').mean()

np.float64(-6.681333772375254)

In [493]:
for item, imp in sorted(zip(list(metrics.columns)+['age'], 
                            pd.concat([metrics, csvTotale.set_index(csvTotale['path'])['age']], axis=1).corr('spearman')['age']), key=lambda x: abs(x[1]), reverse=True)[1:]:
    print(f'{item}: {imp}')

nFrames: 0.6022603562876916
duration: 0.602252908183809
Time 95%  : 0.6016573045219915
duration_90: 0.5994765605409939
duration_5: 0.5948130345006974
Time 75%: 0.5904688371671031
Time 50%: 0.5818011133930301
nVoicedFrames: 0.5644983266317832
Time 25%: 0.5518922891861292
Time 5%: 0.5350617896196377
hnr: -0.5326292343962696
log_hnr: 0.5320784492500874
spectralEnergy250-650: 0.5186342802478199
sm: 0.5010898324353773
fftMean: 0.49037696789958823
spectralEnergy1000-8000: 0.4742084761261652
max_pitch: 0.4694639426920942
min_pitch: -0.46452284591811516
temporalSNR: 0.4389244252671011
mfcc_mean6: -0.4222780140021344
ss: -0.41707174962926924
skew: -0.4170666241622713
mean_pitch: 0.36619218880953386
zcr: 0.3610859442341537
jitter: 0.34920179676737306
mfcc_mean13: -0.34195572156396453
mfcc_mean33: -0.32793932338185033
kurtosis: 0.32287861765139875
sk: 0.3228641469774155
mfcc_mean15: -0.3187933132261265
mfcc_mean17: -0.31683729202774413
mfcc_mean31: -0.2983493012563597
mfcc_mean2: -0.2800573307206

In [494]:
audioEval = {file: librosa.load('.././data/audios_evaluation/'+file, sr=22050)[0] for file in csvEval['path']}

In [495]:
metricsEval = pd.concat([computeMetrics(audioEval), 
                    csvEval.set_index('path')[['hnr', 'shimmer', 'jitter', 'gender', 'max_pitch', 'mean_pitch', 'min_pitch']]], axis=1)
metricsEval['log_hnr']= metricsEval['hnr'].map(lambda x: 20*log10(np.abs(x)))

KeyboardInterrupt: 

In [458]:
ypred = pd.Series(HistGradientBoostingRegressor()
                  .fit(metrics, csvTotale['age'])
                  .predict(metricsEval), name='Predicted', index=csvEval.index)

In [ ]:
from sklearn.metrics import root_mean_squared_error

pd.DataFrame({"nowModel":[
    root_mean_squared_error(pd.read_csv('predicted1501T1827-9100.csv', header=0, index_col=0)['Predicted'], ypred),
    root_mean_squared_error(pd.read_csv('predicted-9650.csv', header=0, index_col=0)['Predicted'], ypred),
    root_mean_squared_error(pd.read_csv('predicted-9308.csv', header=0, index_col=0)['Predicted'], ypred),
    root_mean_squared_error(pd.read_csv('submissionOggi.csv', header=0, index_col=0)['Predicted'], ypred),
    root_mean_squared_error(pd.read_csv('predictedNoHIST-9426.csv', header=0, index_col=0)['Predicted'], ypred),
    root_mean_squared_error(pd.read_csv('predi cted-9322.csv', header=0, index_col=0)['Predicted'], ypred),
    root_mean_squared_error(pd.read_csv('predicted-9235.csv', header=0, index_col=0)['Predicted'], ypred),
    root_mean_squared_error(pd.read_csv('predicted-9196.csv', header=0, index_col=0)['Predicted'], ypred),
    root_mean_squared_error(pd.read_csv('predicted-9150.csv', header=0, index_col=0)['Predicted'], ypred),
    root_mean_squared_error(pd.read_csv('predicted-9319.csv', header=0, index_col=0)['Predicted'], ypred),
    root_mean_squared_error(pd.read_csv('predicted-9525.csv', header=0, index_col=0)['Predicted'], ypred),
    root_mean_squared_error(pd.read_csv('predicted-9309.csv', header=0, index_col=0)['Predicted'], ypred),
    ],
'bestModel':[0,
    root_mean_squared_error(pd.read_csv('predicted1501T1827-9100.csv', header=0, index_col=0)['Predicted'],
                            pd.read_csv('predicted-9650.csv', header=0, index_col=0)['Predicted']),
    root_mean_squared_error(pd.read_csv('predicted1501T1827-9100.csv', header=0, index_col=0)['Predicted'], 
                            pd.read_csv('predicted-9308.csv', header=0, index_col=0)['Predicted']),
    root_mean_squared_error(pd.read_csv('predicted1501T1827-9100.csv', header=0, index_col=0)['Predicted'],
                            pd.read_csv('submissionOggi.csv', header=0, index_col=0)['Predicted']),
    root_mean_squared_error(pd.read_csv('predictedNoHIST-9426.csv', header=0, index_col=0)['Predicted'], 
                            pd.read_csv('9-100.csv', header=0, index_col=0)['Predicted']),
    root_mean_squared_error(pd.read_csv('9-100.csv', header=0, index_col=0)['Predicted'], 
                            pd.read_csv('predicted-9322.csv', header=0, index_col=0)['Predicted']),
    root_mean_squared_error(pd.read_csv('9-100.csv', header=0, index_col=0)['Predicted'],
                            pd.read_csv('predicted-9235.csv', header=0, index_col=0)['Predicted']),
    root_mean_squared_error(pd.read_csv('9-100.csv', header=0, index_col=0)['Predicted'],
                            pd.read_csv('predicted-9196.csv', header=0, index_col=0)['Predicted']),
    root_mean_squared_error(pd.read_csv('9-100.csv', header=0, index_col=0)['Predicted'],
                            pd.read_csv('predicted-9150.csv', header=0, index_col=0)['Predicted']),
    root_mean_squared_error(pd.read_csv('9-100.csv', header=0, index_col=0)['Predicted'],
                            pd.read_csv('predicted-9319.csv', header=0, index_col=0)['Predicted']),
    root_mean_squared_error(pd.read_csv('9-100.csv', header=0, index_col=0)['Predicted'],
                            pd.read_csv('predicted-9525.csv', header=0, index_col=0)['Predicted']),
    root_mean_squared_error(pd.read_csv('9-100.csv', header=0, index_col=0)['Predicted'],
                            pd.read_csv('predicted-9309.csv', header=0, index_col=0)['Predicted']),
    ],
})

,nowModel,bestModel
0,3.458544,0.000000
1,4.067873,3.246054
2,3.846455,2.344467
3,3.704535,3.056776
4,3.443598,3.268457
5,3.974068,3.378994
6,3.891288,3.284294
7,3.607823,3.094170
8,3.836194,3.152830
9,3.837996,2.914613
